# Rainfall Analysis and Planting Advisory

This notebook cleans the Machakos station readings, analyzes rainfall with NumPy, and creates a county summary CSV.

In [1]:
import csv
import numpy as np

counties = ['Nairobi', 'Nakuru', 'Kisumu', 'Mombasa', 'Machakos']
months = ['January', 'February', 'March', 'April', 'May', 'June',
          'July', 'August', 'September', 'October', 'November', 'December']
month_abbreviations = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                       'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],
]
machakos_raw = ['48', '35', '88', 'N/A', '140', ' 20 ',
                '9', '12', '', '45', '182', '96']

In [2]:
def clean_machakos(raw_values, month_labels):
    valid_values = []
    for value in raw_values:
        try:
            valid_values.append(int(value))
        except ValueError:
            continue

    imputed_value = round(np.mean(valid_values))
    cleaned_values = []
    imputed_months = []
    for month, value in zip(month_labels, raw_values):
        try:
            cleaned_values.append(int(value))
        except ValueError:
            cleaned_values.append(imputed_value)
            imputed_months.append(month)

    print(f'Imputed Machakos months: {imputed_months}')
    print(f'Imputed value: {imputed_value} mm')
    return cleaned_values

machakos_clean = clean_machakos(machakos_raw, months)
rainfall = np.array(clean_rows + [machakos_clean])
print(f'Array shape: {rainfall.shape}')
print(f'Array dtype: {rainfall.dtype}')

Imputed Machakos months: ['April', 'September']
Imputed value: 68 mm
Array shape: (5, 12)
Array dtype: int64


In [3]:
annual_totals = rainfall.sum(axis=1)
monthly_averages = rainfall.mean(axis=0)
wettest_month_indices = np.argmax(rainfall, axis=1)
wettest_months = np.array(month_abbreviations)[wettest_month_indices]
dry_month_counts = (rainfall < 50).sum(axis=1)
two_month_totals = rainfall[:, :-1] + rainfall[:, 1:]
planting_window_indices = np.argmax(two_month_totals, axis=1)
planting_windows = np.array([
    f'{month_abbreviations[index]}–{month_abbreviations[index + 1]}'
    for index in planting_window_indices
])

print('Annual totals (mm):')
for county, total in zip(counties, annual_totals):
    print(f'{county}: {total}')
print('\nMonthly averages (mm):')
for month, average in zip(month_abbreviations, monthly_averages):
    print(f'{month}: {average:.1f}')
print('\nWettest month per county:')
for county, month in zip(counties, wettest_months):
    print(f'{county}: {month}')
print('\nDry-month counts (<50 mm):')
for county, count in zip(counties, dry_month_counts):
    print(f'{county}: {count}')
print('\nRecommended planting windows:')
for county, window in zip(counties, planting_windows):
    print(f'{county}: {window}')

Annual totals (mm):
Nairobi: 1045
Nakuru: 916
Kisumu: 1371
Mombasa: 1075
Machakos: 811

Monthly averages (mm):
Jan: 47.0
Feb: 47.6
Mar: 91.0
Apr: 164.4
May: 171.2
Jun: 67.6
Jul: 56.6
Aug: 58.2
Sep: 64.2
Oct: 67.8
Nov: 127.4
Dec: 80.6

Wettest month per county:
Nairobi: Apr
Nakuru: Apr
Kisumu: Apr
Mombasa: May
Machakos: Nov

Dry-month counts (<50 mm):
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6

Recommended planting windows:
Nairobi: Apr–May
Nakuru: Apr–May
Kisumu: Apr–May
Mombasa: Apr–May
Machakos: Nov–Dec


## Planting Advisory

Farmers in Nairobi, Nakuru, Kisumu, and Mombasa should prioritize planting during the April–May window, when consecutive rainfall is highest. Machakos should prioritize November–December, while preparing water-conservation measures for its longer dry periods. Extension teams should send early planting reminders before these windows and drought-preparedness messages to counties with more dry months. The advisory should be combined with local soil, crop, and forecast information before field decisions are made.

## Data Quality Notes

The Machakos station data contained `N/A` and an empty string, so those two months were treated as invalid. Valid values were converted to integers, and each invalid month was imputed with the rounded mean of the valid Machakos readings (68 mm). These results use one year of station data and should be checked against longer-term rainfall records and current forecasts.

In [4]:
with open('rainfall_summary.csv', 'w', newline='', encoding='utf-8') as output_file:
    writer = csv.writer(output_file)
    writer.writerow(['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window'])
    writer.writerows(zip(counties, annual_totals, wettest_months, dry_month_counts, planting_windows))

print('rainfall_summary.csv written with 5 county rows.')

rainfall_summary.csv written with 5 county rows.
